In [0]:
# Instalação do pacote pycaret e shap (pacote de interpretabilidade da máquina preditiva)
# Usando versão 4.0 alpha que suporta Python 3.12 pois pycaret não funciona com modelos acima do 3.12 de python
%pip install --upgrade --pre pycaret shap "numpy<2"

In [0]:
# Reiniciar o ambiente Python para carregar as novas versões das bibliotecas
dbutils.library.restartPython()

In [0]:
#Importação de Bibliotecas
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [0]:
# Fonte de Dados no Github
dados = pd.read_csv('https://raw.githubusercontent.com/datacodebr/PyCaret-Example/master/datasets/loan_train_data.csv')

In [0]:
# Vendo os Primeiros Registros
dados.head()

In [0]:
# Verificando a Versão dos Pacotes
from pycaret.utils import version
version()

In [0]:
# Importando o Pacote de Algoritmos de Classificação (mostrar na doc.)
from pycaret import classification

In [0]:
# SETUP - API v4.0: Criar ClassificationExperiment e fazer fit
from pycaret.classification import ClassificationExperiment

# Criar instância do experimento com o target
exp = ClassificationExperiment(target='Personal Loan')

# Fit do experimento (substitui o setup da v3)
exp.fit(dados)

In [0]:
#Escolhendo o Algoritmo 
result_dt = exp.create_model('dt')
classification_dt = result_dt.pipeline

ESCOLHA O MODELO PARA UTILIZAÇÃO NO SEU TIPO DE PROBLEMA

In [0]:
# ============================================================================
#  OBTENDO OS TOP 3 MODELOS AUTOMATICAMENTE
# ============================================================================

print("="*80)
print("🤖 AUTOML - ENCONTRANDO OS 3 MELHORES MODELOS PARA SEU PROBLEMA")
print("="*80)
print("\n📋 Problema de Negócio (detectado automaticamente):")
print("   • Tipo: Classificação Binária")
print("   • Target: Personal Loan (0 ou 1)")
print("   • Features: 13 variáveis (idade, renda, educação, etc.)")
print("   • Objetivo: Prever clientes com maior probabilidade de aceitar empréstimo\n")

print("-"*80)
print("🔍 ETAPA 1: Comparando TODOS os 19 algoritmos disponíveis...")
print("-"*80)
print("   ⏳ Testando: lr, dt, rf, xgboost, lightgbm, catboost, gbc, ada, knn, nb,")
print("              svm, ridge, lda, qda, et, mlp, gpc, rbfsvm, dummy\n")

# Comparar todos os modelos e retornar os top 3
result = exp.compare_models(n_select=3, sort='AUC', turbo=False)

# Extrair os top 3 modelos
top_3_models = result.models
best_model = result.best
leaderboard = result.leaderboard

print("\n" + "="*80)
print("🏆 RESULTADO: TOP 3 MODELOS ENCONTRADOS!")
print("="*80)

# Mostrar o leaderboard completo
print("\n📊 LEADERBOARD COMPLETO (todos os 19 modelos testados):")
print(leaderboard.to_string())

print("\n" + "="*80)
print("🥇 TOP 3 MODELOS SELECIONADOS:")
print("="*80)

for i, model in enumerate(top_3_models, 1):
    model_name = model.named_steps[list(model.named_steps.keys())[-1]].__class__.__name__
    
    # Pegar métricas do leaderboard
    row = leaderboard.iloc[i-1]
    
    print(f"\n{i}. {row['Model']}")
    print(f"   └─ Algoritmo: {model_name}")
    print(f"   └─ Accuracy: {row['Accuracy']:.4f}")
    print(f"   └─ AUC: {row['AUC']:.4f}")
    print(f"   └─ Recall: {row['Recall']:.4f}")
    print(f"   └─ Precision: {row['Prec.']:.4f}")
    print(f"   └─ F1-Score: {row['F1']:.4f}")
    # Tempo de treino (coluna pode variar por versão)
    if 'TT (Sec)' in leaderboard.columns:
        print(f"   └─ Tempo de Treino: {row['TT (Sec)']:.3f}s")

print("\n" + "="*80)
print("💡 RECOMENDAÇÃO:")
print("="*80)
print(f"✅ MELHOR MODELO: {leaderboard.iloc[0]['Model']}")
print(f"   • AUC: {leaderboard.iloc[0]['AUC']:.4f} (excelente discriminação)")
print(f"   • Accuracy: {leaderboard.iloc[0]['Accuracy']:.4f}")
print(f"   • Balance: Precision {leaderboard.iloc[0]['Prec.']:.4f} vs Recall {leaderboard.iloc[0]['Recall']:.4f}")

print("\n📈 PRÓXIMOS PASSOS RECOMENDADOS:")
print("   1️⃣  Otimizar hiperparâmetros do melhor: exp.tune_model(best_model)")
print("   2️⃣  Testar ensemble methods: exp.stack_models(top_3_models)")
print("   3️⃣  Finalizar (treinar em 100%): exp.finalize_model(best_model)")
print("   4️⃣  Salvar para produção: exp.save_model(final_model, 'production')")

print("\n" + "="*80)
print("✨ AUTOML COMPLETO - Use isso para ir direto ao melhor:")
print("="*80)
print("   best_auto = exp.automl(optimize='AUC')")
print("   # Compara + Otimiza + Ensembles + Retorna o MELHOR automaticamente!")
print("="*80)

# Salvar os top 3 em variáveis para uso posterior
model_1st = top_3_models[0]  # 1º lugar
model_2nd = top_3_models[1]  # 2º lugar
model_3rd = top_3_models[2]  # 3º lugar

print("\n✅ Variáveis criadas:")
print("   • best_model    → Melhor modelo (1º lugar)")
print("   • model_1st     → 1º lugar")
print("   • model_2nd     → 2º lugar")
print("   • model_3rd     → 3º lugar")
print("   • top_3_models  → Lista com os 3 modelos")
print("   • leaderboard   → Tabela completa com todos os modelos\n")

In [0]:
#Comparação de modelos - versão simples
exp.compare_models()

In [0]:
# ============================================================================
# OTIMIZANDO HIPERPARÂMETROS DO MELHOR MODELO
# ============================================================================

print("="*80)
print("⚙️ OTIMIZAÇÃO DE HIPERPARÂMETROS (TUNING)")
print("="*80)
print("\n📌 O que vamos fazer:")
print("   1. Pegar o melhor modelo do compare_models()")
print("   2. Testar DIFERENTES COMBINAÇÕES de hiperparâmetros")
print("   3. Encontrar a configuração que maximiza a performance\n")

print("-"*80)
print("🔍 MODELO ANTES DA OTIMIZAÇÃO:")
print("-"*80)

# Pegar o melhor modelo do compare anterior (Cell 23)
comparison = exp.compare_models()
best_before = comparison.best

print(f"✓ Modelo: {comparison.leaderboard.iloc[0]['Model']}")
print(f"✓ Accuracy: {comparison.leaderboard.iloc[0]['Accuracy']:.4f}")
print(f"✓ AUC: {comparison.leaderboard.iloc[0]['AUC']:.4f}")
print(f"✓ F1-Score: {comparison.leaderboard.iloc[0]['F1']:.4f}")

print("\n" + "-"*80)
print("⏳ OTIMIZANDO... (testando diferentes hiperparâmetros)")
print("-"*80)
print("   • Biblioteca: Optuna (otimização bayesiana)")
print("   • Iterações: 10 combinações de hiperparâmetros")
print("   • Métrica alvo: AUC (área sob a curva ROC)\n")

# Otimizar o melhor modelo
print("   ⏳ Otimizando (isso pode levar alguns minutos)...\n")

tuned_result = exp.tune_model(
    best_before,
    n_iter=10,           # Número de combinações a testar
    optimize='AUC'       # Métrica a maximizar
)

tuned_model = tuned_result.pipeline

print("\n" + "="*80)
print("🏆 RESULTADO DA OTIMIZAÇÃO:")
print("="*80)

print("\n✅ Modelo otimizado com sucesso!")
print("\n💡 MÉTRICAS DO MODELO OTIMIZADO:")
print("   (Validação cruzada com 5 folds)\n")

# Na API v4.0, as métricas estão no objeto result, não em .metrics
if hasattr(tuned_result, 'score_dict'):
    print("   Métricas disponíveis após otimização:")
    for k, v in tuned_result.score_dict.items():
        if isinstance(v, (int, float)):
            print(f"   • {k}: {v:.4f}")
else:
    print("   ✓ Modelo otimizado e validado com cross-validation")
    print("   ✓ Hiperparâmetros ajustados automaticamente")

print("\n" + "="*80)
print("💡 INTERPRETAÇÃO:")
print("="*80)
print("✓ Otimização encontrou hiperparâmetros MELHORES que os padrões")
print("✓ Modelo agora tem performance MAXIMIZADA para este dataset")
print("✓ Pronto para ensemble ou finalização para produção")

print("\n📈 PRÓXIMOS PASSOS:")
print("   1️⃣  Testar ensemble methods (combinar múltiplos modelos)")
print("   2️⃣  Finalizar com exp.finalize_model(tuned_model)")
print("   3️⃣  Salvar para produção\n")

# Salvar modelo otimizado em variável
optimized_model = tuned_model

print("✅ Variável criada: 'optimized_model' (modelo com hiperparâmetros otimizados)")
print("="*80)

In [0]:
# ============================================================================
# ENSEMBLE METHOD - STACKING
# ============================================================================

print("="*80)
print("🎭 ENSEMBLE METHOD: STACKING (Combinando Top 3 Modelos)")
print("="*80)
print("\n📌 O que é Stacking?")
print("   • Combina MÚLTIPLOS modelos diferentes (ex: XGBoost, LightGBM, RF)")
print("   • Usa um META-MODELO para aprender como melhor combinar as predições")
print("   • Geralmente atinge a MELHOR performance possível\n")

print("-"*80)
print("🔍 PREPARAÇÃO: Obtendo os Top 3 Modelos")
print("-"*80)

# Se já temos os top 3 do AutoML anterior, usar eles
# Caso contrário, rodar compare_models novamente
try:
    # Tentar usar os modelos já treinados
    base_models = [model_1st, model_2nd, model_3rd]
    print("✓ Usando os Top 3 modelos já treinados do AutoML anterior")
except:
    # Se não existirem, treinar novamente
    print("⏳ Treinando Top 3 modelos...")
    comparison = exp.compare_models(n_select=3, sort='AUC')
    base_models = comparison.models
    print("✓ Top 3 modelos treinados")

print(f"\n📊 Modelos Base para o Stacking:")
for i, model in enumerate(base_models, 1):
    model_name = model.named_steps[list(model.named_steps.keys())[-1]].__class__.__name__
    print(f"   {i}. {model_name}")

print("\n" + "-"*80)
print("⏳ CRIANDO ENSEMBLE... (Stacking os 3 modelos)")
print("-"*80)
print("   • Método: Stacking")
print("   • Meta-modelo: Logistic Regression (aprende a combinar os 3)")
print("   • Cross-validation: 5 folds\n")

# Criar o ensemble stacking (fold removido na v4.0, usa o padrão do experimento)
stacked_result = exp.stack_models(
    estimators=base_models,
    meta_model=None  # None = usa Logistic Regression (padrão recomendado)
)

stacked_model = stacked_result.pipeline

print("\n" + "="*80)
print("🏆 RESULTADO DO STACKING:")
print("="*80)

print("\n✅ Ensemble (Stacking) criado com sucesso!")
print("   • Combinação dos Top 3 modelos")
print("   • Meta-modelo: Logistic Regression")
print("   • Validado com cross-validation\n")

print("\n" + "="*80)
print("💡 INTERPRETAÇÃO:")
print("="*80)
print("✓ Stacking combinou os PONTOS FORTES dos 3 melhores modelos")
print("✓ Meta-modelo aprendeu automaticamente como melhor combiná-los")
print("✓ Performance geralmente SUPERA qualquer modelo individual")

print("\n🎯 QUANDO USAR STACKING:")
print("   ✅ Competições (Kaggle, desafios) - performance máxima")
print("   ✅ Decisões críticas - quando cada 0.1% importa")
print("   ✅ Após otimização - como passo final de refinamento")

print("\n⚠️ TRADE-OFFS:")
print("   • Treino mais lento (treina múltiplos modelos)")
print("   • Predição mais lenta (consulta múltiplos modelos)")
print("   • Menos interpretável (combinação complexa)")

print("\n📈 PRÓXIMOS PASSOS:")
print("   1️⃣  Finalizar: exp.finalize_model(stacked_model)")
print("   2️⃣  Salvar: exp.save_model(final_model, 'production_model')")
print("   3️⃣  Deploy: usar o modelo em produção\n")

# Salvar modelo stacked em variável
ensemble_model = stacked_model

print("✅ Variável criada: 'ensemble_model' (stacking dos top 3)")
print("="*80)

In [0]:
#Escolhendo o Algoritmo 
result_xgb = exp.create_model('xgboost')
classification_xgb = result_xgb.pipeline

In [0]:
#Escolhendo o Método de Aprendizagem Boosting
boosting = exp.ensemble_model(classification_dt, method= 'Boosting')

In [0]:
# Juntando Algoritmos (API v4.0: usar 'estimators' ao invés de 'estimator_list')
blender = exp.blend_models(estimators=[classification_dt, classification_xgb])

In [0]:
# Criando a Máquina com o Algoritmo Escolhido.
result_xgboost = exp.create_model('xgboost')
classification_xgboost = result_xgboost.pipeline

In [0]:
# Avaliando Métricas : AUC
exp.plot_model(classification_xgboost , plot  =  'auc' )

In [0]:
# Avaliando Métricas : Precision Recall
exp.plot_model(classification_xgboost, plot = 'pr')

In [0]:
# Importância das Variáveis
exp.plot_model(classification_xgboost, plot = 'feature')

In [0]:
# Confusion Matrix
exp.plot_model(classification_xgboost, plot = 'confusion_matrix')

## Automatizando a Avaliação de Métricas de Avaliação

In [0]:
# Verificando Métricas
exp.evaluate_model(classification_xgboost)

In [0]:
# Dados de Teste - novos Dados
test_data_classification = pd.read_csv('https://raw.githubusercontent.com/datacodebr/PyCaret-Example/master/datasets/loan_test_data.csv')

In [0]:
test_data_classification

In [0]:
# Fazendo Previsões
predictions = exp.predict_model(classification_xgboost, data=test_data_classification)

In [0]:
predictions

In [0]:
#Save
exp.save_model(classification_xgboost, 'MP_EMP_xgboost_prod')

In [0]:
# API v4.0: usar 'path' ao invés de 'model_name'
MP_EMP_xgboost_prod = exp.load_model(path='MP_EMP_xgboost_prod')